In [1]:
# Import packages
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import yfinance as yf
from tqdm import tqdm

In [31]:
nflx_chain  = yf.download("NFLX", start="2025-04-02", end="2025-10-02")
spot_chain = yf.download("SPOT", start="2025-04-02", end="2025-10-02")
walt_chain = yf.download("DIS", start="2025-04-02", end="2025-10-02")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [32]:
len(nflx_chain)


126

In [ ]:
n_samples = 100
dt = 1/126
n = 126
n_particles = 500
nflx_close = nflx_chain['Close']
spot_close = spot_chain['Close']
walt_close = walt_chain['Close']


In [ ]:
nflx_returns = np.log(nflx_close / nflx_close.shift(1)).dropna()

mu0 = nflx_returns.mean()
theta0 = nflx_returns.var()
kappa0 = 2.0 # assume mean-reversion speed
sigma0 = 0.3 # assume vol-of-vol
rho0 = -0.7  

sigma_prior_eta = 0.001
mu_prior_eta = 1.00125
tau_prior_eta = 1/sigma_prior_eta**2
lambda_prior = [[10, 0], [0, 5]]
mu_prior = [35e-6, 0.988]
a_prior_sigma = 149
b_prior_sigma = 0.025


In [ ]:
#Prior Params
lambda_prior = 0.15


In [ ]:
def convertCDF(v, V_sort, W_sort, n_particles):

    W_sort = W_sort / np.sum(W_sort)
    N = len(V_sort)

    if v < V_sort[0]:
        return 0.0

    if v > V_sort[-1]:
        return 1.0

    j = np.searchsorted(V_sort, v) - 1
    j = max(0, min(j, N-2))

    vj = V_sort[j]
    vj1 = V_sort[j+1]

    if j == 0:  # first interval
        weight = W_sort[0] + 0.5 * W_sort[1]
        return (v - vj)/(vj1 - vj) * weight

    elif j == N-2:  # last interval
        base = np.sum(W_sort[:N-2]) + 0.5 * W_sort[N-2]
        weight = 0.5 * W_sort[N-2] + W_sort[N-1]
        return base + (v - vj)/(vj1 - vj) * weight

    else:  # middle intervals
        base = np.sum(W_sort[:j]) + 0.5 * W_sort[j]
        weight = 0.5 * W_sort[j] + 0.5 * W_sort[j+1]
        return base + (v - vj)/(vj1 - vj) * weight

In [ ]:
def get_vol_estimator(cdf, n_particles, V_sort):

    U = np.random.rand(n_particles)
    particles = np.zeros(n_particles)

    N = len(V_sort)

    for i, u in enumerate(U):

        j = np.searchsorted(cdf, u)
        j = min(j, N-2)

        C_prev = 0 if j == 0 else cdf[j-1]

        v1 = V_sort[j]
        v2 = V_sort[j+1]

        particles[i] = v1 + (v2 - v1) * (u - C_prev) / (cdf[j] - C_prev)

    return particles.mean()

In [ ]:
def gibbs_sampler(m_b, L_b, a_s, b_s, n_iter):

    beta_samples = np.zeros((n_iter, len(m_b)))
    sigma_samples = np.zeros(n_iter)

    # initial value
    sigma_samples[0] = 1.0

    for i in range(1, n_iter):

        sigma_prev = sigma_samples[i-1]

        # sample beta_i | sigma_prev
        cov_b = sigma_prev**2 * np.linalg.inv(L_b)
        beta_i = np.random.multivariate_normal(m_b, cov_b)

        beta_samples[i] = beta_i

        # sample sigma_i | beta_i

        sigma2_i = 1 / np.random.gamma(shape=a_s, scale=1/b_s)
        sigma_samples[i] = np.sqrt(sigma2_i)

    return beta_samples, sigma_samples

In [ ]:
V = np.full(n_particles, theta0)
weights = np.ones(n_particles) / n_particles
theta = theta0

mu_estimates = []
kappa_estimates = []
theta_estimates = []
sigma_estimates = []
rho_estimates = []
lambda_estimates = []
v_estimates = []

for i in range(n_samples):  
    V = np.full(n_particles, theta0)  # initialize particle variances
    
    # Estimate v(k*dt) , i.e. volatility at each time step
    for k in range(1, n):  
        R_k = nflx_returns[k]  

        epsilon = np.random.normal(0, 1, n_particles)
        z = (R_k - mu0*dt) / (np.sqrt(dt) * np.sqrt(np.maximum(V, 1e-8))) # to prevent V from being 0 
        w = rho0 * z + np.sqrt(1 - rho0**2) * epsilon
        V = np.maximum(V + kappa0*(theta - V)*dt + sigma0*np.sqrt(V*dt)*w, 0)

        W = (1/np.sqrt(2*np.pi*V*dt)) * np.exp(-0.5 * ((R_k - mu0*dt-1)**2) / (V*dt))
        W = W/np.sum(W)

        U = np.column_stack(V, W)

        sort_idx = np.argsort(V)
        V_sorted = V[sort_idx]
        W_sorted = W[sort_idx]

        cdf = np.array([convertCDF(v, V_sorted, W_sorted, n_particles)
                for v in V_sorted])
        v_estimate = get_vol_estimator(cdf, n_particles, V_sorted)
        v_estimates.append(v_estimate)
    v_estimates.append(v_estimates[-1]) # Assume v(n-1) = v(n) for  a sufficiently dense time discretisation grid.

    # Estimate mu
    y_S = ((nflx_returns / np.sqrt(v_estimates))/ np.sqrt(dt)).reshape(-1,1).T
    x_S = ((1/np.sqrt(v_estimates)) / np.sqrt(dt)).reshape(-1,1).T

    tau_eta = x_S.T @ x_S + tau_prior_eta
    eta_hat = np.linalg.inv(x_S.T @ x_S) * x_S.T * y_S
    mu_eta = (x_S.T @ x_S * eta_hat + mu_prior_eta * tau_prior_eta) / tau_eta 
    eta_i = np.random.normal(loc=mu_eta, scale=1/np.sqrt(tau_eta)) 
    mu_i = (eta_i -1)/dt 
    mu_estimates.append(mu_i)

    # Estimate kappa, theta, sigma 
    y_v = (v_estimates[1:]/np.sqrt(v_estimates[:-1])).reshape(-1,1) / np.sqrt(dt)
    x1_v = (1/v_estimates[:-1]).reshape(-1,1) / np.sqrt(dt)
    x2_v = np.sqrt(v_estimates[:-1]).reshape(-1,1) / np.sqrt(dt)
    X_v = np.hstack((x1_v, x2_v))

    lambda_beta = X_v.T @ X_v + lambda_prior
    b_hat = np.linalg.inv(X_v.T @ X_v) * X_v.T @ y_v
    mu_beta = np.linalg.inv(lambda_beta) * (lambda_prior @ mu_prior + X_v.T @ X_v * b_hat)

    a_sigma = a_prior_sigma + n/2
    b_sigma = b_prior_sigma + 0.5(y_v.T @ y_v + 1/mu_prior @ lambda_prior @ mu_prior - mu_beta.T @ lambda_beta @ mu_beta) #scalar

    # Use gibbs sampling since sigma and beta distributions are not independent
    beta_estimates, sigma_estimates = gibbs_sampler(y_v, X_v, mu_beta, lambda_beta, a_sigma, b_sigma, n_iter=1000)

    kappa_estimates = (1 - beta_estimates[:,1])/dt
    theta_estimates = beta_estimates[:,0]/(1-beta_estimates[:,1])